In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt

In [17]:
names = []
with open('names.txt', 'r') as f:
    for name in f.readlines():
        names.append(name[:-1])

## Implementing the MLP as per Bengio's paper
This is a charecter level architecture. The model takes in the trigram as input and predicts the next word.

In [26]:
# create an index from charecter to index an reverse
special_char = '.'
chars = set(list(''.join(names)) + [special_char])
stoi = {c:i for i, c in enumerate(sorted(chars))}
itos = {i:c for c, i in stoi.items()}
num_chars = len(stoi)

In [27]:
stoi

{'.': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26}

In [55]:
# create the trigrams and the targets
X = []
Y = []
block_size = 3
for name in names:
    name = [special_char] + list(name) + [special_char]
    for i in range(1, len(name)):
        end_idx = i
        start_idx = end_idx - block_size
        input_block = [max(idx, 0) for idx in range(start_idx, end_idx)]
        target_char = name[end_idx]
        input_ngram = [name[idx] for idx in input_block]
        Y.append(stoi[target_char])
        X.append([stoi[c] for c in input_ngram])
        # print(''.join(input_ngram) , "-->", target_char)


In [56]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [57]:
print(f'Number of training example = {len(X)}')

Number of training example = 228145


In [58]:
# split the data into train, val and test
train_frac = 0.9
val_frac = 0.05
test_frac = 0.05
n1  = int(len(X)*train_frac)
n2 = n1 + int(len(X)*val_frac)

print(f'Number of train example = {n1}')
print(f'Number of val example = {n2 - n1}')
print(f'Number of test example = {len(X) - n2}')

Xtr, Ytr = X[:n1], Y[:n1]
Xval, Yval = X[n1:n2], Y[n1:n2]
Xtest, Ytest = X[n2:], Y[n2:]

Number of train example = 205330
Number of val example = 11407
Number of test example = 11408


### Build a neural network on the lines of Bengio's paper


In [116]:
# initialize the nn
num_emb_dim = 2
C = torch.randn((num_chars, num_emb_dim))
W1 = torch.randn((block_size*num_emb_dim, 100))
b1 = torch.randn(100)
W2 = torch.randn((100, num_chars))

In [117]:
x = X[:2]
x.shape

torch.Size([2, 3])

In [118]:
C

tensor([[-0.8499, -0.7876],
        [ 0.1473, -1.7025],
        [ 1.1781,  1.0168],
        [ 0.3355,  1.3333],
        [ 1.6154,  0.2633],
        [-1.1301,  0.7347],
        [-0.0755, -0.4311],
        [ 1.8525, -1.2351],
        [ 1.0521,  0.2346],
        [ 0.3848,  0.2952],
        [-0.0796, -2.1398],
        [ 1.1780, -0.8355],
        [-0.1755,  0.4264],
        [-0.3801,  0.4818],
        [-0.8335, -0.5745],
        [ 0.1312, -0.1049],
        [-1.2812, -0.8418],
        [ 1.0252,  0.4447],
        [-0.3922,  1.7981],
        [ 0.5122,  0.7870],
        [-1.3596, -0.3224],
        [ 0.4731, -2.1573],
        [-0.6631,  0.5536],
        [ 0.0464, -1.2137],
        [ 2.1225,  0.4668],
        [ 0.0910, -0.8122],
        [ 0.4582, -0.3190]])

In [119]:
x1 = C[x]
x1.shape # batch size, ngram, emb_dim

torch.Size([2, 3, 2])

In [120]:
# concatenate
x2 = x1.view(x1.shape[0], block_size*num_emb_dim)

In [121]:
x2.shape

torch.Size([2, 6])

In [122]:
W1.shape

torch.Size([6, 100])

In [123]:
(x2@W1).shape

torch.Size([2, 100])

In [ ]:
b1.shape

torch.Size([100])

In [126]:
(x2@W1 + b1).shape

torch.Size([2, 100])

In [127]:
# transform the embedding output
x2.shape
x3 = (x2@W1 + b1)
# apply the non linearity
x4 = torch.tanh(x3)

In [133]:
# apply next transformation to output the logits
x5 = x3@W2
# apply the softmax to output the probability
output = F.softmax(x5, dim=-1)

In [ ]:
# compiling the forward pass together
x = X[:2]
# concatenate
x2 = x1.view(x1.shape[0], block_size*num_emb_dim)
# transform the embedding output
x2.shape
x3 = (x2@W1 + b1)
# apply the non linearity
x4 = torch.tanh(x3)
# apply next transformation to output the logits
x5 = x3@W2
# apply the softmax to output the probability
output = F.softmax(x5, dim=-1)